In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Demo").master("local[*]").getOrCreate()

# XGBoost
We'll use in this notebook an external library

## Load Data & Preparation
Let's load the clean Airbnb dataset in again , but this time we will split in 3, so we have a validation set as well as a training and test sets
We created it in a previous notebook, it should exists in `/home/jovyan/work/datasets/outpus/airbnb/clean_data` 
Also, let's index all of our categorical features, and set our label to be **`log(price)`**.

In [5]:
from pyspark.sql.functions import log, col
from pyspark.ml.feature import StringIndexer, VectorAssembler

file_path = "/home/jovyan/work/datasets/output/airbnb/clean_data"
airbnb_df = spark.read.parquet(file_path)
train_df, test_df = airbnb_df.withColumn("label", log("price")).randomSplit([.8, .2], seed=42)

#Select all categorical columns and index them 
categorical_cols = [field for (field, dataType) in train_df.dtypes if dataType == "string"]
index_output_cols = [x + "Index" for x in categorical_cols]

string_indexer = StringIndexer(inputCols=categorical_cols, outputCols=index_output_cols, handleInvalid="skip")

#Select all numeric columns except price and label
numeric_cols = [field for (field, dataType) in train_df.dtypes if ((dataType == "double")& (field != "price") & (field != "label"))]

assembler_inputs = index_output_cols + numeric_cols
vec_assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")

### Distributed Training of XGBoost Models
We create an SparkXGBRegressor with the following params as a json
* n_estimators: 100
* learning_rate: 0.1
* max_depth: 4
* random_state:42
* missing:0

In [6]:
from xgboost.spark import SparkXGBRegressor
from pyspark.ml import Pipeline

params = {"n_estimators": 100, "learning_rate": 0.1, "max_depth": 4, "random_state": 42, "missing": 0}

xgboost = SparkXGBRegressor(**params)

pipeline = Pipeline(stages=[string_indexer, vec_assembler, xgboost])
pipeline_model = pipeline.fit(train_df)

2026-04-03 10:43:32,459 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cpu', 'learning_rate': 0.1, 'max_depth': 4, 'random_state': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': 0.0}
2026-04-03 10:43:37,072 INFO XGBoost-PySpark: _fit Finished xgboost training!


## Evaluate Model Performance
* Remember to exponentiate the label column so we can evaluate the real prediction
* Set the exp(prediction) in a new column called just prediction

In [8]:
from pyspark.sql.functions import exp, col

log_pred_df = pipeline_model.transform(test_df)

exp_xgboost_df = log_pred_df.withColumn("prediction", exp(col("prediction")))

exp_xgboost_df.select("price", "prediction").show()

+-----+------------------+
|price|        prediction|
+-----+------------------+
|185.0| 139.7594610550372|
|100.0| 105.1140536810815|
|230.0| 156.4149120077575|
|110.0|109.50080616517614|
| 90.0|149.21984447868232|
|122.0|127.81038399086017|
|130.0|148.16779403781229|
| 58.0| 84.19411444259131|
|109.0|125.95907471265838|
|144.0| 186.9170115917517|
|160.0| 84.14133784068082|
|232.0| 152.8634909563706|
| 56.0| 72.88620736211803|
|140.0|118.95464825667585|
|200.0| 166.2115247770347|
|165.0|166.58587472129184|
|175.0|143.00357703317144|
| 80.0|  82.9631449910508|
|199.0|156.74126058308096|
|172.0|171.42640888571154|
+-----+------------------+
only showing top 20 rows



In [9]:
from pyspark.ml.evaluation import RegressionEvaluator

regression_evaluator = RegressionEvaluator(predictionCol="prediction", labelCol="price", metricName="rmse")

rmse = regression_evaluator.evaluate(exp_xgboost_df)
r2 = regression_evaluator.setMetricName("r2").evaluate(exp_xgboost_df)
print(f"RMSE is {rmse}")
print(f"R2 is {r2}")

RMSE is 36.165604586229904
R2 is 0.6077907553652846
